# Text to 3D AI Model - Colab Host

Run this notebook in Google Colab with a GPU runtime. It clones the GitHub repository, installs dependencies, downloads the TripoSR checkpoint, starts the FastAPI app, and exposes it with ngrok.

## 1. Clone the GitHub repository

In [ ]:
REPO_URL = "https://github.com/LucyAlex12/Text_to_3d_ai_model.git"

!rm -rf Text_to_3d_ai_model
!git clone {REPO_URL}
%cd Text_to_3d_ai_model

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt pyngrok huggingface_hub

## 3. Download TripoSR weights

`model.ckpt` is too large for normal Git commits, so this cell downloads it into `TripoSR/`.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

Path("TripoSR").mkdir(exist_ok=True)

for filename in ["config.yaml", "model.ckpt"]:
    hf_hub_download(
        repo_id="stabilityai/TripoSR",
        filename=filename,
        local_dir="TripoSR"
    )

print("TripoSR checkpoint ready")

## 4. Configure ngrok

Recommended: add your ngrok authtoken to Colab secrets as `NGROK_AUTH_TOKEN`. If it is not set, this cell asks you to paste it.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = None

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok configured")

## 5. Start the public web app

The default Colab settings below use Stable Diffusion 1.5 at 512x512 to reduce GPU memory pressure. Change `IMAGE_MODEL_KIND` to `sdxl` only if your runtime has enough VRAM.

In [ ]:
import os
import subprocess
import time
import requests
from pyngrok import ngrok

os.environ["IMAGE_MODEL_KIND"] = "sd15"
os.environ["SDXL_WIDTH"] = "512"
os.environ["SDXL_HEIGHT"] = "512"
os.environ["SDXL_STEPS"] = "20"
os.environ["SDXL_GUIDANCE_SCALE"] = "7.0"
os.environ["TRIPOSR_MAX_MC_RESOLUTION"] = "256"

ngrok.kill()

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(120):
    try:
        requests.get("http://127.0.0.1:8000/progress", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Server did not start. Check the logs below.")

public_url = ngrok.connect(8000, "http").public_url
print("Open this public URL:")
print(public_url)
print("\nLeave this runtime running while you use the app.")

## Optional: view server logs

In [ ]:
while True:
    line = server.stdout.readline()
    if not line:
        break
    print(line, end="")